In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

print("Basic Text Embeddings and Semantic Similarity:")

try:
    # Load a lightweight, fast model for demonstrations
    model = SentenceTransformer('all-MiniLM-L6-v2')
    print(f"✅ Loaded model: {model._modules['0'].auto_model.config.name_or_path}")
    
    # Sample sentences for similarity testing
    sentences = [
        "The weather is beautiful today",
        "It's a gorgeous day outside", 
        "I love programming in Python",
        "Python is my favorite programming language",
        "The cat is sleeping on the couch"
    ]
    
    # Generate embeddings
    embeddings = model.encode(sentences)
    print(f"Generated embeddings shape: {embeddings.shape}")
    print(f"Embedding dimension: {embeddings.shape[1]}")
    
    # Calculate similarity matrix
    similarities = model.similarity(embeddings, embeddings)
    
    # Display similarity results
    print("\nSemantic Similarity Matrix:")
    for i, sent1 in enumerate(sentences):
        for j, sent2 in enumerate(sentences):
            if i < j:  # Only show upper triangle
                similarity_score = similarities[i][j].item()
                print(f"  '{sent1[:30]}...' ↔ '{sent2[:30]}...': {similarity_score:.3f}")
    
    # Find most similar pair
    max_similarity = 0
    most_similar_pair = (0, 0)
    for i in range(len(sentences)):
        for j in range(i + 1, len(sentences)):
            if similarities[i][j] > max_similarity:
                max_similarity = similarities[i][j].item()
                most_similar_pair = (i, j)
    
    print(f"\nMost similar pair (score: {max_similarity:.3f}):")
    print(f"  1: {sentences[most_similar_pair[0]]}")
    print(f"  2: {sentences[most_similar_pair[1]]}")

except Exception as e:
    print(f"⚠️ Example requires sentence-transformers package: {e}")
    print("Note: In practice, this would load pre-trained models and compute embeddings")
    raise AssertionError(
        f"sentence_transformers example (cell 0) failed: {type(e).__name__}: {e}"
    ) from e

print("✅ Basic text embeddings and semantic similarity completed")

## Example 2: Semantic Search Implementation

In [ ]:
# Example 2: Semantic Search Implementation
from sentence_transformers import SentenceTransformer, util
import torch

print("Semantic Search Implementation:")

try:
    # Initialize model for semantic search
    model = SentenceTransformer('all-MiniLM-L6-v2')
    
    # Document corpus for search
    corpus = [
        "Python is a high-level programming language known for its simplicity",
        "Machine learning is a subset of artificial intelligence",
        "Data science involves extracting insights from large datasets", 
        "Web development includes both frontend and backend programming",
        "Cloud computing provides on-demand access to computing resources",
        "Cybersecurity protects systems and networks from digital attacks",
        "Software engineering is the application of engineering principles to software",
        "Database management systems store and organize large amounts of data",
        "Artificial intelligence enables machines to simulate human intelligence",
        "DevOps combines development and operations for faster delivery"
    ]
    
    # Encode corpus once for efficient searching
    corpus_embeddings = model.encode(corpus, convert_to_tensor=True)
    print(f"Encoded {len(corpus)} documents into embeddings")
    
    # Search queries
    queries = [
        "What is AI and machine learning?",
        "Programming languages for beginners", 
        "How to secure computer systems?"
    ]
    
    # Perform semantic search for each query
    for query in queries:
        print(f"\n🔍 Query: '{query}'")
        
        # Encode query
        query_embedding = model.encode(query, convert_to_tensor=True)
        
        # Compute cosine similarity
        hits = util.semantic_search(query_embedding, corpus_embeddings, top_k=3)
        
        # Display top results
        for i, hit in enumerate(hits[0]):
            doc_id = hit['corpus_id']
            similarity_score = hit['score']
            document = corpus[doc_id]
            
            print(f"  {i+1}. [Score: {similarity_score:.3f}] {document}")
    
    # Demonstrate batch search
    print(f"\n📊 Batch Search Results:")
    query_embeddings = model.encode(queries, convert_to_tensor=True)
    
    # Fast batch semantic search
    all_hits = util.semantic_search(query_embeddings, corpus_embeddings, top_k=2)
    
    for query_idx, query_hits in enumerate(all_hits):
        avg_score = sum(hit['score'] for hit in query_hits) / len(query_hits)
        print(f"  Query {query_idx + 1}: avg relevance = {avg_score:.3f}")

except Exception as e:
    print(f"⚠️ Semantic search demonstration: {e}")
    print("Note: Would implement full semantic search with ranking and relevance scoring")
    raise AssertionError(
        f"sentence_transformers example (cell 2) failed: {type(e).__name__}: {e}"
    ) from e

print("✅ Semantic search implementation completed")

## Example 3: Text Clustering and Topic Analysis

In [ ]:
# Example 3: Text Clustering and Topic Analysis
from sentence_transformers import SentenceTransformer
import numpy as np
from collections import defaultdict

print("Text Clustering and Topic Analysis:")

try:
    # Load model for clustering
    model = SentenceTransformer('all-MiniLM-L6-v2')
    
    # Sample texts from different topics
    texts = [
        # Technology/Programming
        "Python programming is essential for data science",
        "JavaScript frameworks make web development easier",
        "Machine learning algorithms improve with more data",
        
        # Health/Medicine  
        "Regular exercise improves cardiovascular health",
        "A balanced diet provides essential nutrients",
        "Meditation reduces stress and anxiety levels",
        
        # Finance/Business
        "Stock market investments require careful analysis",
        "Diversification reduces portfolio risk significantly", 
        "Cryptocurrency adoption is growing worldwide",
        
        # Travel/Geography
        "Paris is known for its beautiful architecture",
        "Mountain climbing requires proper equipment and training",
        "Beach destinations are popular for summer vacations"
    ]
    
    # Generate embeddings for clustering
    embeddings = model.encode(texts)
    print(f"Generated embeddings for {len(texts)} texts")
    
    # Simple clustering using cosine similarity
    def simple_clustering(embeddings, threshold=0.6):
        """Cluster texts based on similarity threshold"""
        clusters = []
        assigned = set()
        
        for i, emb1 in enumerate(embeddings):
            if i in assigned:
                continue
                
            cluster = [i]
            assigned.add(i)
            
            for j, emb2 in enumerate(embeddings):
                if j != i and j not in assigned:
                    # Calculate cosine similarity
                    similarity = np.dot(emb1, emb2) / (np.linalg.norm(emb1) * np.linalg.norm(emb2))
                    if similarity > threshold:
                        cluster.append(j)
                        assigned.add(j)
            
            clusters.append(cluster)
        
        return clusters
    
    # Perform clustering
    clusters = simple_clustering(embeddings, threshold=0.4)
    
    print(f"\nFound {len(clusters)} clusters:")
    
    # Display clusters with their texts
    for cluster_id, text_indices in enumerate(clusters):
        print(f"\n📂 Cluster {cluster_id + 1} ({len(text_indices)} texts):")
        
        cluster_texts = [texts[i] for i in text_indices]
        for text in cluster_texts:
            print(f"  • {text}")
        
        # Calculate cluster centroid and find most representative text
        if len(text_indices) > 1:
            cluster_embeddings = embeddings[text_indices]
            centroid = np.mean(cluster_embeddings, axis=0)
            
            # Find text closest to centroid
            distances = [np.linalg.norm(emb - centroid) for emb in cluster_embeddings]
            representative_idx = text_indices[np.argmin(distances)]
            print(f"  🎯 Representative: {texts[representative_idx]}")
    
    # Analyze cluster quality
    def analyze_cluster_quality(clusters, embeddings):
        """Calculate intra-cluster similarity"""
        if len(clusters) <= 1:
            return 0.0
            
        total_intra_similarity = 0
        total_pairs = 0
        
        for cluster in clusters:
            if len(cluster) < 2:
                continue
                
            for i in range(len(cluster)):
                for j in range(i + 1, len(cluster)):
                    emb1, emb2 = embeddings[cluster[i]], embeddings[cluster[j]]
                    similarity = np.dot(emb1, emb2) / (np.linalg.norm(emb1) * np.linalg.norm(emb2))
                    total_intra_similarity += similarity
                    total_pairs += 1
        
        return total_intra_similarity / total_pairs if total_pairs > 0 else 0
    
    quality_score = analyze_cluster_quality(clusters, embeddings)
    print(f"\n📊 Clustering Quality (avg intra-cluster similarity): {quality_score:.3f}")

except Exception as e:
    print(f"⚠️ Clustering demonstration: {e}")
    print("Note: Would perform advanced clustering with scikit-learn KMeans or hierarchical clustering")
    raise AssertionError(
        f"sentence_transformers example (cell 4) failed: {type(e).__name__}: {e}"
    ) from e

print("✅ Text clustering and topic analysis completed")

## Example 4: Cross-Encoder Reranking and Advanced Scoring

In [ ]:
# Example 4: Cross-Encoder Reranking and Advanced Scoring
from sentence_transformers import SentenceTransformer
from sentence_transformers.cross_encoder import CrossEncoder

print("Cross-Encoder Reranking and Advanced Scoring:")

try:
    # Initialize bi-encoder for initial retrieval
    bi_encoder = SentenceTransformer('all-MiniLM-L6-v2')
    
    # Initialize cross-encoder for reranking (more accurate but slower)
    cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L6-v2')
    print("✅ Loaded bi-encoder and cross-encoder models")
    
    # Query and candidate documents
    query = "How to improve software development productivity?"
    
    documents = [
        "Agile methodologies help teams deliver software faster and more efficiently",
        "Code reviews improve software quality and reduce bugs in production", 
        "Automated testing saves time and catches errors early in development",
        "Continuous integration enables faster deployment and better collaboration",
        "Documentation helps new team members understand complex codebases",
        "Pair programming increases code quality and knowledge sharing",
        "The weather forecast shows rain for the weekend",
        "Machine learning models require large amounts of training data",
        "Proper project management tools streamline development workflows"
    ]
    
    # Stage 1: Initial retrieval with bi-encoder
    print(f"\n🔍 Query: '{query}'")
    print("Stage 1: Bi-encoder retrieval")
    
    query_embedding = bi_encoder.encode(query)
    doc_embeddings = bi_encoder.encode(documents)
    
    # Calculate cosine similarities
    from sentence_transformers import util
    similarities = util.cos_sim(query_embedding, doc_embeddings)[0]
    
    # Get top-k candidates for reranking
    top_k = 5
    top_indices = similarities.argsort(descending=True)[:top_k]
    
    print(f"Top {top_k} candidates from bi-encoder:")
    bi_encoder_results = []
    for i, idx in enumerate(top_indices):
        score = similarities[idx].item()
        doc = documents[idx]
        bi_encoder_results.append((idx.item(), score, doc))
        print(f"  {i+1}. [Score: {score:.3f}] {doc[:60]}...")
    
    # Stage 2: Reranking with cross-encoder
    print(f"\nStage 2: Cross-encoder reranking")
    
    # Prepare query-document pairs for cross-encoder
    query_doc_pairs = [[query, documents[idx]] for idx, _, _ in bi_encoder_results]
    
    # Get cross-encoder scores
    cross_scores = cross_encoder.predict(query_doc_pairs)
    
    # Sort by cross-encoder scores
    reranked_results = []
    for i, (orig_idx, bi_score, doc) in enumerate(bi_encoder_results):
        cross_score = cross_scores[i]
        reranked_results.append((orig_idx, bi_score, cross_score, doc))
    
    # Sort by cross-encoder score (descending)
    reranked_results.sort(key=lambda x: x[2], reverse=True)
    
    print("Final reranked results:")
    for i, (orig_idx, bi_score, cross_score, doc) in enumerate(reranked_results):
        print(f"  {i+1}. [Cross: {cross_score:.3f}, Bi: {bi_score:.3f}] {doc[:60]}...")
    
    # Compare ranking changes
    print(f"\n📊 Ranking Analysis:")
    original_order = [result[0] for result in bi_encoder_results]
    reranked_order = [result[0] for result in reranked_results]
    
    ranking_changes = 0
    for i, (orig_pos, new_pos) in enumerate(zip(original_order, reranked_order)):
        if orig_pos != new_pos:
            ranking_changes += 1
    
    print(f"Documents with changed rankings: {ranking_changes}/{len(reranked_results)}")
    
    # Demonstrate confidence scoring
    print(f"\nConfidence Analysis:")
    max_cross_score = max(result[2] for result in reranked_results)
    min_cross_score = min(result[2] for result in reranked_results)
    score_range = max_cross_score - min_cross_score
    
    print(f"Score range: {min_cross_score:.3f} to {max_cross_score:.3f} (range: {score_range:.3f})")
    
    # Flag highly relevant vs irrelevant documents
    for i, (_, bi_score, cross_score, doc) in enumerate(reranked_results):
        if cross_score > 0.5:
            relevance = "🟢 High"
        elif cross_score > 0.0:
            relevance = "🟡 Medium" 
        else:
            relevance = "🔴 Low"
        print(f"  {relevance} relevance: {doc[:50]}...")

except Exception as e:
    print(f"⚠️ Cross-encoder demonstration: {e}")
    print("Note: Would use cross-encoder models for improved ranking accuracy")
    raise AssertionError(
        f"sentence_transformers example (cell 6) failed: {type(e).__name__}: {e}"
    ) from e

print("✅ Cross-encoder reranking and advanced scoring completed")

## Example 5: Multilingual Models and Production NLP Pipeline

In [ ]:
# Example 5: Multilingual Models and Production NLP Pipeline
from sentence_transformers import SentenceTransformer
import time
from typing import List, Dict

print("Multilingual Models and Production NLP Pipeline:")

try:
    # Load multilingual model for cross-language tasks
    multilingual_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
    print("✅ Loaded multilingual sentence transformer model")
    
    # Production NLP pipeline class
    class ProductionNLPPipeline:
        def __init__(self, model):
            self.model = model
            self.cache = {}
            self.stats = {
                "requests": 0,
                "cache_hits": 0,
                "avg_processing_time": 0,
                "total_time": 0
            }
        
        def encode_with_caching(self, texts: List[str], cache_key: str = None) -> Dict:
            """Encode texts with caching for production efficiency"""
            start_time = time.time()
            self.stats["requests"] += 1
            
            # Use cache if available
            if cache_key and cache_key in self.cache:
                self.stats["cache_hits"] += 1
                return {
                    "embeddings": self.cache[cache_key],
                    "processing_time": 0.001,  # Minimal cache retrieval time
                    "cache_hit": True
                }
            
            # Encode texts
            embeddings = self.model.encode(texts, show_progress_bar=False)
            processing_time = time.time() - start_time
            
            # Update stats
            self.stats["total_time"] += processing_time
            self.stats["avg_processing_time"] = self.stats["total_time"] / self.stats["requests"]
            
            # Cache results
            if cache_key:
                self.cache[cache_key] = embeddings
            
            return {
                "embeddings": embeddings,
                "processing_time": processing_time,
                "cache_hit": False
            }
        
        def get_performance_stats(self) -> Dict:
            """Get pipeline performance statistics"""
            cache_hit_rate = (self.stats["cache_hits"] / self.stats["requests"]) * 100 if self.stats["requests"] > 0 else 0
            return {
                **self.stats,
                "cache_hit_rate": cache_hit_rate,
                "cache_size": len(self.cache)
            }
    
    # Initialize production pipeline
    pipeline = ProductionNLPPipeline(multilingual_model)
    
    # Multilingual text samples
    multilingual_texts = {
        "English": ["Artificial intelligence is transforming our world", "Technology helps solve complex problems"],
        "Spanish": ["La inteligencia artificial está transformando nuestro mundo", "La tecnología ayuda a resolver problemas complejos"],
        "French": ["L'intelligence artificielle transforme notre monde", "La technologie aide à résoudre des problèmes complexes"],
        "German": ["Künstliche Intelligenz verändert unsere Welt", "Technologie hilft dabei, komplexe Probleme zu lösen"],
        "Chinese": ["人工智能正在改变我们的世界", "技术有助于解决复杂问题"]
    }
    
    print(f"\n🌍 Multilingual Semantic Analysis:")
    
    # Process each language and find cross-language similarities
    language_embeddings = {}
    for language, texts in multilingual_texts.items():
        cache_key = f"lang_{language.lower()}"
        result = pipeline.encode_with_caching(texts, cache_key)
        language_embeddings[language] = result["embeddings"]
        
        status = "🚀 (cached)" if result["cache_hit"] else f"⚙️ ({result['processing_time']:.3f}s)"
        print(f"  {language}: {len(texts)} texts processed {status}")
    
    # Cross-language similarity analysis
    print(f"\n🔗 Cross-Language Similarity Analysis:")
    
    # Compare first sentence across all languages (should be high similarity)
    base_text = "AI transforming world"
    similarities = {}
    
    for lang, embeddings in language_embeddings.items():
        first_sentence_embedding = embeddings[0:1]  # First sentence embedding
        
        # Calculate similarity with other languages' first sentences
        for other_lang, other_embeddings in language_embeddings.items():
            if lang != other_lang:
                other_first_embedding = other_embeddings[0:1]
                similarity = multilingual_model.similarity(first_sentence_embedding, other_first_embedding)[0][0].item()
                pair_key = f"{lang}-{other_lang}"
                similarities[pair_key] = similarity
    
    # Display top cross-language similarities
    sorted_similarities = sorted(similarities.items(), key=lambda x: x[1], reverse=True)
    print("Top cross-language similarities:")
    for pair, similarity in sorted_similarities[:5]:
        print(f"  {pair}: {similarity:.3f}")
    
    # Demonstrate production features
    print(f"\n🏭 Production Pipeline Features:")
    
    # Test caching by re-processing same texts
    print("Testing cache performance...")
    for language in ["English", "Spanish"]:
        texts = multilingual_texts[language]
        cache_key = f"lang_{language.lower()}"
        
        # Should hit cache this time
        result = pipeline.encode_with_caching(texts, cache_key)
        print(f"  {language} re-processing: {'🚀 Cache hit' if result['cache_hit'] else '⚙️ Cache miss'}")
    
    # Performance statistics
    stats = pipeline.get_performance_stats()
    print(f"\nPipeline Performance:")
    print(f"  Total requests: {stats['requests']}")
    print(f"  Cache hits: {stats['cache_hits']}")
    print(f"  Cache hit rate: {stats['cache_hit_rate']:.1f}%")
    print(f"  Average processing time: {stats['avg_processing_time']:.3f}s")
    print(f"  Cache size: {stats['cache_size']} items")
    
    # Demonstrate batch processing efficiency
    print(f"\n⚡ Batch Processing Demo:")
    all_texts = []
    for texts in multilingual_texts.values():
        all_texts.extend(texts)
    
    batch_result = pipeline.encode_with_caching(all_texts, "batch_all_languages")
    print(f"Processed {len(all_texts)} texts in {batch_result['processing_time']:.3f}s")
    print(f"Average time per text: {batch_result['processing_time']/len(all_texts)*1000:.1f}ms")

except Exception as e:
    print(f"⚠️ Multilingual pipeline demonstration: {e}")
    print("Note: Would use multilingual models for cross-language semantic understanding")
    raise AssertionError(
        f"sentence_transformers example (cell 8) failed: {type(e).__name__}: {e}"
    ) from e

print("✅ Multilingual models and production NLP pipeline completed")